In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import pandas as pd


def detect_and_remove_outliers(h5_file_path):
    def remove_single_outlier(series):
        """Remove the single largest outlier dynamically."""
        if len(series) <= 2:
            return series, []  # Not enough data to determine outliers
        median = series.median()
        deviations = abs(series - median)
        threshold = 50 * deviations.median()  # Dynamic threshold for small arrays
        max_deviation_idx = deviations.idxmax()
        if deviations[max_deviation_idx] > threshold:
            removed = [max_deviation_idx]  # Return index, not value
            cleaned = series.drop(max_deviation_idx)
        else:
            removed = []
            cleaned = series
        return cleaned, removed

    outliers = {}
    with h5py.File(h5_file_path, 'r') as h5_file:
        for gene in h5_file.keys():
            outliers[gene] = {}
            for sigma_group in h5_file[gene]:
                dataset = h5_file[gene][sigma_group]
                corr_matrix = np.array(dataset)

                between_corrs = pd.Series(corr_matrix[np.triu_indices_from(corr_matrix, k=1)])
                within_corrs = pd.Series(np.diag(corr_matrix))

                # Dynamically remove the largest outlier
                between_cleaned, removed_between = remove_single_outlier(between_corrs)
                within_cleaned, removed_within = remove_single_outlier(within_corrs)

                # # Log removed and remaining values
                # print(f"Gene: {gene}, Sigma: {sigma_group}")
                # print(f"  Removed Between Outlier Index: {removed_between}")
                # print(f"  Remaining Between Correlations: {between_cleaned.tolist()}")
                # print(f"  Removed Within Outlier Index: {removed_within}")
                # print(f"  Remaining Within Correlations: {within_cleaned.tolist()}")

                outliers[gene][sigma_group] = {
                    "between_outliers": removed_between,  # Store indices, not values
                    "within_outliers": removed_within
                }
    return outliers

def plot_avg_correlation_with_violin(sigma_results, output_folder=''):
    """
    Plot violin plots of average correlation with scatter points showing the full distribution.
    A line is added to link the medians for each sigma group.

    Parameters:
    - sigma_results (dict): Dictionary {sigma: list of correlation values}.
    - output_folder (str): Path to save the plot. If empty, the plot is displayed.
    """
    # Prepare data for plotting
    sigmas = sorted(sigma_results.keys())
    data = []
    labels = []
    medians = []  # To store median values for the line plot

    for sigma in sigmas:
        values = sigma_results[sigma]
        data.extend(values)
        labels.extend([sigma] * len(values))
        medians.append(np.median(values))  # Compute median for each sigma group
    
    # Set up the figure
    plt.figure(figsize=(10, 7))
    sns.set(style="whitegrid")
    palette = sns.color_palette("viridis", len(sigmas))  # Color palette for violins
    
    # Violin plot
    sns.violinplot(x=labels, y=data, palette=palette, inner=None, linewidth=1.5, alpha=0.7)
    
    # Scatter points over the violins
    for i, sigma in enumerate(sigmas):
        x_vals = np.random.normal(loc=i, scale=0.05, size=len(sigma_results[sigma]))  # Add jitter for visibility
        plt.scatter(x_vals, sigma_results[sigma], color='black', s=20, alpha=0.7, zorder=3)
    
    # Line linking the medians
    plt.plot(range(len(sigmas)), medians, color='red', marker='o', linestyle='-', linewidth=2, markersize=8, label='Median')

    # Labels and Title
    plt.title('Average Correlation per Sigma Value', fontsize=16, fontweight='bold')
    plt.xlabel('Sigma Value', fontsize=14, fontweight='bold')
    plt.ylabel('Average Correlation', fontsize=14, fontweight='bold')
    plt.xticks(ticks=range(len(sigmas)), labels=[str(sigma) for sigma in sigmas], fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # Legend
   # plt.legend(fontsize=12)

    # Save or display the plot
    if output_folder is None or output_folder == '':
        plt.show()
    else:
        output_file_path = os.path.join(output_folder, 'Average_correlation_violin_plot_with_median_line.png')
        plt.savefig(output_file_path, bbox_inches='tight', dpi=300)
        print(f'Figure saved to {output_file_path}')
        plt.show()

def compute_avg_correlation_per_sigma(h5_file_path, outliers_detected):
    sigma_results = {}
    with h5py.File(h5_file_path, 'r') as h5_file:
        for gene in h5_file.keys():
            for sigma_group in h5_file[gene]:
                sigma_value = float(sigma_group.split('_')[1])
                corr_matrix = np.array(h5_file[gene][sigma_group])

                between_corrs = pd.Series(corr_matrix[np.triu_indices_from(corr_matrix, k=1)])
                within_corrs = pd.Series(np.diag(corr_matrix))

                outlier_indices_between = outliers_detected[gene][sigma_group]['between_outliers']
                outlier_indices_within = outliers_detected[gene][sigma_group]['within_outliers']

                between_cleaned = between_corrs.drop(outlier_indices_between)
                within_cleaned = within_corrs.drop(outlier_indices_within)

                avg_corr = np.nanmean(between_cleaned)
                if sigma_value not in sigma_results:
                    sigma_results[sigma_value] = []
                sigma_results[sigma_value].append(avg_corr)
    return sigma_results

def main(h5_file_path,output_folder=''):
    # Step 1: Detect and remove outliers
    outliers = detect_and_remove_outliers(h5_file_path)
    print("Outliers detected successfully.")

    # Step 2: Compute average correlations per sigma
    sigma_results = compute_avg_correlation_per_sigma(h5_file_path, outliers)
    print("Average correlations calculated successfully.")

    # Step 3: Plot average correlations with standard deviations
    plot_avg_correlation_with_violin(sigma_results, output_folder)

# Run analysis
h5_file_folder = r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\zBrain\parallel_analysis\202412_full\correlation_matrices'
h5_file_path =os.path.join(h5_file_folder, 'cross_correlation_results.h5')

main(h5_file_path,h5_file_folder)
